# Initial-Condition Audit & Campaign Analysis

This notebook covers the initial-condition modules in `workflows/diagnostics/` (`inventory_and_hash`, `compare_netcdf_structure`, `compute_ic_statistics`, `plot_component_differences`, `check_physical_consistency`, `summarize_campaign`, configured by `ic_config.yaml`) — a 6-step IC audit pipeline. It explores differences in initialization states (IC differences) and checks physical consistency between coupled model components.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
import json
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

ENVIRONMENT_SHARE = Path(sys.prefix) / 'share'
os.environ.setdefault('PROJ_DATA', str(ENVIRONMENT_SHARE / 'proj'))
os.environ.setdefault('GDAL_DATA', str(ENVIRONMENT_SHARE / 'gdal'))

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = (
    [Path(_repo_override).expanduser().resolve()]
    if _repo_override
    else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
)
REPO_ROOT = next((p for p in _repo_candidates if (p / "workflows" / "diagnostics" / "ic_config.yaml").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "workflows" / "diagnostics" / "ic_config.yaml"


## Configuration

In [2]:
# =========================================================================
# USER CONTROL PANEL
# =========================================================================

# CONFIG_PATH is set above
PILOT_ONLY = True
SINGLE_DATE = None
FILTER_COMPONENTS = None
COMPUTE_HASH = True

# Durable products go to <s2d_diag_root>/multimodel/initial_conditions and figures
# are published flat as fig_ic_*.png under the drift figure root (env_paths).
from esp_lab import env_paths
FIGURE_OUTDIR = env_paths.drift_figure_root()
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)
# =========================================================================

## Step 1 — Inventory & Hash

In [3]:
%%time
from workflows.diagnostics.inventory_and_hash import run as run_step1

manifest_df = run_step1(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE, compute_hash=COMPUTE_HASH)
display(manifest_df.head())


IC Analysis — Step 1: Inventory and Hash
  ref  : /global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/BruteForce
  test : /global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/JRA55-FOSIRL
  dates: ['1980-05-01-00000']
  hash : True

  [REUSE] 1980-05-01-00000 → /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/initial_conditions/paired_manifests/1980-05-01-00000.csv

  Combined manifest → /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/initial_conditions/paired_manifests/combined_manifest.json

--- Audit Summary ---
component  status   
atm        DIFFERENT    10
cpl        IDENTICAL     1
ice        DIFFERENT     1
lnd        IDENTICAL     1
ocn        DIFFERENT     1
rof        IDENTICAL     1



/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/workflows/diagnostics/inventory_and_hash.py:145: UserWarning: match_start_dates: 69 date(s) exist only in ref (e.g. 1984-02-01-00000); skipping.
  all_pairs = match_start_dates(ref_root, test_root)
/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/workflows/diagnostics/inventory_and_hash.py:145: UserWarning: match_start_dates: 1 date(s) exist only in test (e.g. 2019-05-01-00000); skipping.
  all_pairs = match_start_dates(ref_root, test_root)


,start_date,component,member,status,ref_path,test_path,ref_size_bytes,test_size_bytes,ref_sha256,test_sha256,ref_timestamp,test_timestamp,ref_nc_dims,test_nc_dims
0,1980-05-01-00000,atm,EN00,DIFFERENT,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,2648427740,2648427744,47910e15762b931403dd46acf56143d7096312ac229f83...,a93f8ca4a03754a6a73429c4e8d8feb334ceb37f5dfd10...,1980-05-01 00:00:00,1980-05-01 00:00:00,"{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i...","{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i..."
1,1980-05-01-00000,atm,EN01,DIFFERENT,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,2648427740,2648427744,a0c79798e2efdb158498fa523ead0b0c359652826bbd73...,02aadbf9f169489365cca991b9e5fea5250935150eb83f...,1980-05-01 00:00:00,1980-05-01 00:00:00,"{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i...","{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i..."
2,1980-05-01-00000,atm,EN02,DIFFERENT,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,2648427740,2648427744,422c8574a322ed05793e6af415cb46c00273f7296e90b8...,7decbc26e6c300b43a411b7ad2b526933defdb55abdd83...,1980-05-01 00:00:00,1980-05-01 00:00:00,"{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i...","{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i..."
3,1980-05-01-00000,atm,EN03,DIFFERENT,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,2648427740,2648427744,467d38e6fb2570abbcb4a2772a8710b73bec7eb6f5d07e...,6b3d3e00937d6bc591651981bd0218025e1136aa37fffc...,1980-05-01 00:00:00,1980-05-01 00:00:00,"{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i...","{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i..."
4,1980-05-01-00000,atm,EN04,DIFFERENT,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSIN...,2648427740,2648427744,6434e52ef6178a42e211ab61471015a90cd08f40dd776b...,7ea07eada60ffe85a3b2ecb7b8127662ed8b716b92c7bc...,1980-05-01 00:00:00,1980-05-01 00:00:00,"{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i...","{'ncol_d': 48602, 'ncol': 21600, 'lev': 80, 'i..."


CPU times: user 24.7 ms, sys: 23.2 ms, total: 47.9 ms
Wall time: 212 ms


## Step 2 — Compare NetCDF Structure

In [ ]:
%%time
from workflows.diagnostics import compare_netcdf_structure

class_df = compare_netcdf_structure.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE)
display(class_df.head())


IC Analysis — Step 2: NetCDF Structure Comparison

  Date: 1980-05-01-00000  (12 DIFFERENT/INCOMPATIBLE rows)
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mism

,start_date,component,variable,location,classification
0,1980-05-01-00000,atm,BVRIM,common,unknown
1,1980-05-01-00000,atm,C10H16,common,unknown
2,1980-05-01-00000,atm,C2H4,common,unknown
3,1980-05-01-00000,atm,C2H5O2,common,unknown
4,1980-05-01-00000,atm,C2H5OOH,common,unknown


CPU times: user 1.59 s, sys: 509 ms, total: 2.1 s
Wall time: 9.3 s


: 

## Step 3 — Compute Variable-Level IC Statistics

In [ ]:
%%time
from workflows.diagnostics import compute_ic_statistics

stats_df = compute_ic_statistics.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE, filter_components=FILTER_COMPONENTS)
display(stats_df.head())


IC Analysis — Step 3: Compute Variable-Level IC Statistics

  [1980-05-01-00000] atm EN00
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN01
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN02
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN03
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN04
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN05
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN06
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN07
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN08
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN09
    cosp_t

## Step 4 — Plot Component Differences

In [ ]:
%%time
from workflows.diagnostics import plot_component_differences

plot_component_differences.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE, filter_components=FILTER_COMPONENTS, figure_outdir=FIGURE_OUTDIR)
print(f"Output figures saved to {FIGURE_OUTDIR}")


## Step 5 — Cross-Component Physical Consistency

In [ ]:
%%time
from workflows.diagnostics import check_physical_consistency

consistency_df = check_physical_consistency.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE)
display(consistency_df.head())


## Step 6 — Campaign Summary

In [ ]:
%%time
from workflows.diagnostics import summarize_campaign
from IPython.display import Image, display

campaign_df = summarize_campaign.run(
    config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, figure_outdir=FIGURE_OUTDIR
)
display(campaign_df.head())

img_path = FIGURE_OUTDIR / summarize_campaign.campaign_summary_figure_name()
if img_path.exists():
    display(Image(filename=str(img_path)))


## Validation

In [ ]:
assert not manifest_df.empty, "Step 1 returned empty results"
assert not class_df.empty, "Step 2 returned empty results"
assert not stats_df.empty, "Step 3 returned empty results"
assert not consistency_df.empty, "Step 5 returned empty results"
assert not campaign_df.empty, "Step 6 returned empty results"
print("All validation assertions passed.")
